# Análisis de parámetros: características de forma, estructura radial y textura
**PSIB 2026 Q2 — Grupo 9 (Del Carril, Escobar, Uthurralt)**

Continúa a `pipeline_2_segmentacion.ipynb` (etapas 1 a 4): en vez de volver a correr esas etapas, esta celda carga sus resultados ya guardados en `resultados/etapaN/<CONJUNTO>/` (`cargar_etapa2`, `cargar_etapa4`), así que primero hay que haber corrido ese notebook con el mismo `CONJUNTO`.
| Etapa | Qué hace | Archivo |
|---|---|---|
| 5 | Características y textura (GLCM) en borde vs. centro | `etapas/etapa5.py` |
| 6 | Decisión estadística: comparación entre clases y elección del umbral  | *pendiente* |
| 7 | Evaluación en testeo | *pendiente* |

In [27]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from etapas.datos import listar_dataset
from etapas import e2_mejoramiento as e2
from etapas import e4_morfologia as e4

RUTA_DATASET = Path("Dataset_reducido")
CONJUNTO = "Entrenamiento"   # tiene que ser el mismo CONJUNTO con el que se corrió pipeline_2_segmentacion.ipynb
RUTA_RESULTADOS = Path("resultados")

df = listar_dataset(RUTA_DATASET, CONJUNTO)
print(f"Conjunto: {CONJUNTO} — {len(df)} imágenes")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Conjunto: Entrenamiento — 37 imágenes


### Carga de resultados guardados (etapas 2 y 4)

In [28]:
carpeta_e2 = RUTA_RESULTADOS / "etapa2" / CONJUNTO
resultados_e2 = {id_: e2.cargar_etapa2(carpeta_e2 / f"{id_}_e2.npz") for id_ in df.index}

carpeta_e4 = RUTA_RESULTADOS / "etapa4" / CONJUNTO
res_e4 = {id_: e4.cargar_etapa4(carpeta_e4 / f"{id_}_e4.npz") for id_ in df.index}

print(f"Cargadas etapa2 y etapa4 de {len(df)} imágenes desde {RUTA_RESULTADOS.resolve()}")

Cargadas etapa2 y etapa4 de 37 imágenes desde C:\Users\delca\Downloads\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\resultados


## 5. Características

Elegimos 6 parametros para analizar las heridas, parametros que tienen que ver con la forma, con la estructura y con las texturas.

### 5.1 Descriptores de forma

Funciones en `etapas/forma.py`, aplicadas sobre la lesión final (`res_e4[id_]["lesion"]`: la
máscara rellena recortada a la piel, la misma que usa la etapa 4 para definir zonas).

- **Circularidad** = 4·π·Área / Perímetro² del contorno externo. 1 = círculo perfecto; baja con
  anillos incompletos o lesiones alargadas.
- **Dimensión fractal del borde**: pendiente de log(N cajas) vs. log(1/tamaño de caja) sobre el
  contorno de 1 px (box-counting). Cerca de 1 = borde liso; más alto = borde festoneado — esperable
  en el anillo descamado de la TC.

Las lesiones anotadas como "difusa" (ocupan toda la foto) no tienen un borde real que medir:
`dim_fractal_borde` queda en NaN para esos casos, igual que no se genera banda de `borde` en la etapa 4.

In [29]:
from etapas import etapa5 as e5

descriptores = {id_: e5.descriptores_forma(res_e4[id_]["lesion"]) for id_ in df.index}
df_forma = pd.DataFrame.from_dict(descriptores, orient="index")
df_forma.index.name = "id"

carpeta_forma = RUTA_RESULTADOS / "forma" / CONJUNTO
ruta_forma = e5.guardar_csv(df_forma, carpeta_forma / "descriptores_forma.csv")
print(f"Guardados descriptores de forma de {len(df_forma)} imágenes en {ruta_forma.resolve()}")

Guardados descriptores de forma de 37 imágenes en C:\Users\delca\Downloads\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\resultados\forma\Entrenamiento\descriptores_forma.csv


### 5.2 Estructura radial: ratio de eritema y perfil de gradiente radial

Dos formas de medir lo mismo (si el eritema se concentra en el borde o está parejo), una como
número único y otra como curva. Usan `a_suave` (etapa 2) y las zonas `borde`/`centro`/`lesion`
de `res_e4` (etapa 4). Funciones en `etapas/forma.py`.

- **Ratio de eritema** $R_e = \bar{I}_{borde}(a^*) \,/\, \bar{I}_{centro}(a^*)$: $R_e > 1$ =
  borde más rojo que el centro (anillo); $R_e \approx 1$ = placa pareja.
- **Perfil de gradiente radial** $I(r)$: media de a* en bandas de distancia al centroide de la
  lesión, de r=0 (centroide) a r=1 (punto más alejado dentro de la lesión). Es la versión
  continua del mismo contraste: en una TC se espera que suba hacia r=1.

Mismo cuidado que con R_e: a* puede ser negativo, así que si la media del centro da ~0 el
cociente no tiene mucho sentido (queda en NaN); en ese caso conviene mirar el perfil completo.

In [30]:
df_radial = e5.procesar_dataset_radial(
    canales_a={id_: resultados_e2[id_]["a_suave"] for id_ in df.index},
    lesiones={id_: res_e4[id_]["lesion"] for id_ in df.index},
    bordes={id_: res_e4[id_]["borde"] for id_ in df.index},
    centros={id_: res_e4[id_]["centro"] for id_ in df.index},
    n_bins=10,
)

carpeta_radial = RUTA_RESULTADOS / "estructura_radial" / CONJUNTO
ruta_radial = e5.guardar_csv(df_radial, carpeta_radial / "estructura_radial.csv")
print(f"Guardados descriptores de estructura radial de {len(df_radial)} imágenes en {ruta_radial.resolve()}")


Guardados descriptores de estructura radial de 37 imágenes en C:\Users\delca\Downloads\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\resultados\estructura_radial\Entrenamiento\estructura_radial.csv


### 5.4 Textura: entropía y contraste GLCM (borde vs. centro)

Funciones en `etapas/textura.py`. Usan `L` **sin filtrar** de la etapa 2 (`resultados_e2[id_]["L"]`,
no `L_suave` -- la mediana borra justo las escamas/pápulas que la GLCM tiene que medir) dentro de
`borde` y `centro` (etapa 4). La GLCM se calcula por separado en cada zona, contando solo pares de
píxeles donde los dos extremos están dentro de la misma región (si no, un par borde-centro
contaminaría la textura de una zona con la de la otra).

$$ENT = -\sum_{i,j} P(i,j)\log_2 P(i,j) \qquad CON = \sum_{i,j} (i-j)^2 \, P(i,j)$$

Hipótesis de la tabla: en TC, $ENT_{borde} > ENT_{centro}$ (desorganización por escamas/pápulas
en el anillo) y $CON_{borde} \gg CON_{centro}$ (contraste local alto en esa franja). Abajo se
calculan `ent_diff` y `con_diff` (borde − centro) y se chequea esa hipótesis directamente: en
cuántas imágenes de cada clase el signo da el esperado.

In [31]:
df_textura = e5.procesar_dataset_textura(
    canales_L={id_: resultados_e2[id_]["L"] for id_ in df.index},
    bordes={id_: res_e4[id_]["borde"] for id_ in df.index},
    centros={id_: res_e4[id_]["centro"] for id_ in df.index},
)

carpeta_textura = RUTA_RESULTADOS / "textura" / CONJUNTO
ruta_textura = e5.guardar_csv(df_textura, carpeta_textura / "textura.csv")
print(f"Guardados descriptores de textura de {len(df_textura)} imágenes en {ruta_textura.resolve()}")

Guardados descriptores de textura de 37 imágenes en C:\Users\delca\Downloads\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\PSIB2026Q2-TPI-Grupo9-DelCarril-Ecobar-Uthurralt-Descriptores\resultados\textura\Entrenamiento\textura.csv


## 6. Decisión estadistica

Con los parametros encontrados realizamos una decisión para encontrar los umbrales y luego decidir si la lesión es una tiña corpis o no

Primero encontramos la estadistica de los parametros para la tiña y para la no tiña

In [32]:
from etapas import etapa6 as e6

cols_forma = ["circularidad", "dim_fractal_borde"]
cols_radial = ["ratio_eritema"] + [f"gradiente_r{i}" for i in range(10)]
cols_textura = ["ent_borde", "ent_centro", "ent_diff", "con_borde", "con_centro", "con_diff"]

df_stats = (df[["clase"]]
            .join(df_forma[cols_forma])
            .join(df_radial[cols_radial])
            .join(df_textura[cols_textura]))

tabla_forma = e6.resumen_por_clase(df_stats, cols_forma)
tabla_radial = e6.resumen_por_clase(df_stats, cols_radial)
tabla_textura = e6.resumen_por_clase(df_stats, cols_textura)

e6.guardar_csv(tabla_forma, RUTA_RESULTADOS / "estadistica" / CONJUNTO / "forma.csv")
e6.guardar_csv(tabla_radial, RUTA_RESULTADOS / "estadistica" / CONJUNTO / "radial.csv")
e6.guardar_csv(tabla_textura, RUTA_RESULTADOS / "estadistica" / CONJUNTO / "textura.csv")

print("Forma:")
display(tabla_forma.round(3))
print("\nEstructura radial (ratio de eritema + perfil de gradiente radial, banda por banda):")
display(tabla_radial.round(3))
print("\nTextura:")
display(tabla_textura.round(3))

Forma:


clase              Otras                                      TC         \
estadistica       desvio maximo  media mediana minimo   n desvio maximo   
descriptor                                                                
circularidad       0.084  0.885  0.795   0.824  0.505  24  0.054  0.881   
dim_fractal_borde  0.016  1.029  1.004   1.008  0.965  24  0.012  1.032   

clase                                        
estadistica        media mediana minimo   n  
descriptor                                   
circularidad       0.816   0.831  0.671  13  
dim_fractal_borde  1.010   1.009  0.983  12


Estructura radial (ratio de eritema + perfil de gradiente radial, banda por banda):


clase          Otras                                       TC                 \
estadistica   desvio  maximo  media mediana minimo   n desvio  maximo  media   
descriptor                                                                     
ratio_eritema  0.472   1.678  0.970   1.044 -0.477  23  0.219   1.603  1.178   
gradiente_r0   6.364  28.070  8.269   6.270  1.195  24  4.056  14.584  8.024   
gradiente_r1   6.090  26.721  8.351   6.980 -0.899  24  4.049  15.445  8.112   
gradiente_r2   5.497  23.509  8.453   7.116 -1.865  24  3.987  15.316  8.320   
gradiente_r3   5.154  22.314  8.627   7.725 -1.904  24  3.578  14.729  8.597   
gradiente_r4   4.777  20.903  8.829   8.636 -1.141  24  3.291  14.635  8.952   
gradiente_r5   4.923  19.662  8.696   8.300 -1.700  24  3.348  14.809  9.086   
gradiente_r6   4.820  18.772  8.488   8.415 -1.478  24  3.368  15.035  9.226   
gradiente_r7   4.643  17.911  8.196   8.405 -1.289  24  3.358  15.633  9.402   
gradiente_r8   4.560  18.145  8.087   8.437 -1.693  24  3.365  15.649  9.588   
gradiente_r9   4.738  16.995  8.347   8.925 -1.685  24  3.656  16.797  9.091   

clase                             
estadistica   mediana minimo   n  
descriptor                        
ratio_eritema   1.084  0.931  12  
gradiente_r0    7.560  0.923  13  
gradiente_r1    7.568  1.333  13  
gradiente_r2    7.893  1.686  13  
gradiente_r3    8.287  2.269  13  
gradiente_r4    9.116  3.140  13  
gradiente_r5    8.909  3.149  13  
gradiente_r6    9.172  3.483  13  
gradiente_r7    9.376  3.846  13  
gradiente_r8    9.838  4.105  13  
gradiente_r9    9.163  3.297  13


Textura:


clase        Otras                                      TC                \
estadistica desvio maximo  media mediana minimo   n desvio maximo  media   
descriptor                                                                 
ent_borde    0.539  6.220  4.741   4.634  3.734  23  0.589  5.451  4.256   
ent_centro   0.597  6.010  4.859   4.881  3.862  24  0.620  4.985  4.051   
ent_diff     0.366  0.661 -0.108  -0.170 -0.875  23  0.366  0.782  0.245   
con_borde    0.715  3.523  0.995   0.740  0.324  23  0.274  1.160  0.500   
con_centro   0.658  2.680  1.181   0.927  0.312  24  0.216  0.884  0.471   
con_diff     0.442  1.243 -0.180  -0.223 -1.248  23  0.119  0.276  0.039   

clase                           
estadistica mediana minimo   n  
descriptor                      
ent_borde     4.389  3.416  12  
ent_centro    4.139  2.677  13  
ent_diff      0.356 -0.598  12  
con_borde     0.480  0.164  12  
con_centro    0.428  0.148  13  
con_diff      0.005 -0.100  12

### 6.3 Alternativa: regla de votos (6 criterios, 3 niveles)

Nuestra forma de hacer la decisión, pensada a partir de la hipótesis de partida del trabajo (anillo activo en tiña: borde más irregular, más rojo y más desorganizado que el centro; forma más redondeada) en vez de un solo descriptor: revisar 6 criterios y contar cuántos se cumplen.

1. `circularidad` > mediana de Entrenamiento
2. `dim_fractal_borde` > mediana de Entrenamiento
3. `ratio_eritema` > 1
4. perfil radial: banda más externa (borde) > banda más interna (centro)
5. `ent_diff` (entropía borde − centro) > 0
6. `con_diff` (contraste borde − centro) > 0

Cuando se cumple que **4 o menos → "es tiña"**, **3 o menos → "no es"** . Código y detalle completo en `etapas/etapa6.py`. Esta regla se decidió debido a que es preferible 

In [ ]:
from etapas import decision_votos as e6v

cols_criterios = ["circularidad", "dim_fractal_borde", "ratio_eritema",
                  "gradiente_r0", "gradiente_r9", "ent_diff", "con_diff"]
df_votos = (df[["clase"]]
            .join(df_forma[["circularidad", "dim_fractal_borde"]])
            .join(df_radial[["ratio_eritema", "gradiente_r0", "gradiente_r9"]])
            .join(df_textura[["ent_diff", "con_diff"]]))

salida_votos = e6v.clasificar_dataset(df_votos)
resultado_votos = e6v.evaluar(salida_votos["veredicto"], df["clase"])
display(salida_votos["cumplidos"].value_counts().sort_index())

print(f"\nTabla: clase real x veredicto\n")
display(resultado_votos["tabla"])

print(f"\nAccuracy:              {resultado_votos['accuracy']:.1%}")
print(f"Sensibilidad (TC):      {resultado_votos['sensibilidad_tc']:.1%}")
print(f"Especificidad (Otras):  {resultado_votos['especificidad_otras']:.1%}")

Umbral: 4 o más de 6 criterios -> predice TC

Cuántos criterios cumple cada imagen (de 0 a 6):


cumplidos
0     1
1     4
2    10
3    10
4     6
5     6
Name: count, dtype: int64


Tabla: clase real x veredicto



veredicto,es tiña,no es
clase,,
Otras,3,21
TC,9,4



Accuracy:              81.1%
Sensibilidad (TC):      69.2%
Especificidad (Otras):  87.5%


Se puede observar que el codigo es bueno para detectar tiña, pero se le escapan algunas tiñas y algunas que no son tiña las detecta como  que son, este error se podria corregir en un futuro con algoritmos de machine learning o tecnicas avanzadas.